<a href="https://colab.research.google.com/github/vencov/FAV_course/blob/main/TutSACF03/TutSACF_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gammatone filterbank -> IHC -> SACF pitch model

A compact periodicity-pitch pipeline:

1. **Gammatone filterbank** (ERB-spaced channels) -- the peripheral auditory filter bank.
2. **Simple IHC model** -- half-wave rectification, a ~1 kHz lowpass (loss of phase-locking
   at high frequencies), and a compressive power-law nonlinearity.
3. **SACF** (summary autocorrelation function) -- per-channel autocorrelation, summed
   across channels, used to estimate the fundamental frequency (F0) from periodicity.

All model functions are in the single cell below; the demos (static F0, then a melody with
time-varying F0) follow in the cells after it.



### Task: Autocorrelation function

The next step is to calculate the **autocorrelation function** of each IHC output channel. The autocorrelation measures the similarity of a signal with a delayed version of itself and is therefore useful for detecting periodicity in the signal.

For a discrete signal $x[n]$, the autocorrelation at lag $k$ can be calculated directly as

$$
R_{xx}[k] = \sum_n x[n]\,x[n+k].
$$

For an efficient implementation, we will use the **Wiener--Khinchin theorem**, which states that the autocorrelation can be obtained from the inverse Fourier transform of the squared magnitude of the Fourier transform:

$$
R_{xx}[k]
=
\mathcal{F}^{-1}
\left\{
X(f)X^*(f)
\right\}[k],
$$

where

$$
X(f) = \mathcal{F}\{x[n]\}
$$

is the Fourier transform of the signal and $X^*(f)$ is its complex conjugate.

To obtain the **linear** autocorrelation rather than the circular autocorrelation introduced by the FFT, the signal has to be zero-padded before calculating the FFT. Choose the FFT length such that

$$
N_{\mathrm{FFT}} \geq 2N,
$$

where $N$ is the length of the input signal.

#### Your task

Complete the missing parts of the function below:

```python
def autocorr_fft(x, max_lag):
    """Autocorrelation of 1-D array x (lags 0..max_lag), via the
    Wiener-Khinchin theorem."""

    n = len(x)

    # Choose the FFT length
    nfft = 1
    while nfft < 2 * n:
        nfft *= 2

    # TODO, STUDENTS' TASK FOR 15 POINTS, DETAILS IN THE PDF
    # Calculate the Fourier transform
    #X = ...
    # Calculate the autocorrelation using the Wiener-Khinchin theorem
    #acf =

    return acf[:max_lag + 1]
```

Your implementation should return the autocorrelation values for lags

$$
k = 0,1,\ldots,\texttt{max\_lag}.
$$

**Hint:** In NumPy, the relevant functions are `np.fft.fft()`, `np.fft.ifft()`, and `np.conj()`.
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, butter

%matplotlib inline


# ---------------- Gammatone filterbank (gammaf1.m / getGFBFilterCoefs.m) ----------------
def gammaf1(fs, f0, beta, n, ftype):
    """Digital gammatone filter (n-th order) as a complex-coefficient IIR.
    ftype = 1: bilinear transform, 2: impulse invariance."""
    if ftype == 1:
        w0 = 2 * np.pi * f0 / fs
        W0 = np.tan(w0 / 2)
        Beta = np.tan(np.pi * beta / fs)
        tmp = 1 + 1 / Beta - 1j * W0 / Beta
        b = np.array([1, 1], dtype=complex) / tmp
        a = np.array([tmp, 1 - 1 / Beta - 1j * W0 / Beta]) / tmp
    elif ftype == 2:
        b = np.array([1 - np.exp(-2 * np.pi * beta / fs)], dtype=complex)
        a = np.array([1, -np.exp(-(2 * np.pi * beta + 1j * 2 * np.pi * f0) / fs)])
    else:
        raise ValueError("ftype must be 1 or 2")
    b2, a2 = np.array([1.0 + 0j]), np.array([1.0 + 0j])
    for _ in range(n):                      # cascade n identical 1st-order sections
        b2 = np.convolve(b, b2)
        a2 = np.convolve(a, a2)
    return b2, a2


def getGFBFilterCoefs(fs, centerfreq, bw):
    centerfreq = np.atleast_1d(centerfreq)
    beta = 1.0183 * np.atleast_1d(bw)       # 4th-order gammatone (0.637 would be 2nd order)
    b, a = [], []
    for k in range(len(centerfreq)):
        bt, at = gammaf1(fs, centerfreq[k], beta[k], 4, 2)
        b.append(bt); a.append(at)
    return np.vstack(b), np.vstack(a)


class BMParams:
    """Container replacing the MATLAB struct CBM."""
    pass


def initBMGTpar(cfArray, fs):
    cfArray = np.asarray(cfArray, dtype=float)
    bandwidth = 24.7 + cfArray / 9.265      # ERB (Glasberg & Moore)
    CBM = BMParams()
    CBM.GT_b, CBM.GT_a = getGFBFilterCoefs(fs, cfArray, bandwidth)
    CBM.bandwidth = bandwidth
    CBM.cfArray = cfArray
    return CBM


def fce_BMgammatone(signal, par, pastGTn=None):
    """Filter `signal` through the gammatone filterbank.
    Returns out (n_samples x n_channels, HIGH CF in column 0) and filter states."""
    signal = np.asarray(signal).ravel()
    nsect = len(par.cfArray)
    out = np.zeros((len(signal), nsect))
    order = par.GT_a.shape[1] - 1
    if pastGTn is None or np.size(pastGTn) == 0:
        pastGTn = np.zeros((order, nsect), dtype=complex)
    for ch in range(nsect):
        outBM, pastGTn[:, ch] = lfilter(par.GT_b[ch], par.GT_a[ch], signal, zi=pastGTn[:, ch])
        out[:, ch] = 2 * np.real(outBM)
    return out[:, ::-1], pastGTn            # fliplr: highest freq -> lowest-index section


def erb_space(f_low, f_high, n):
    """n center frequencies spaced uniformly on the ERB scale (Moore & Glasberg 1983)."""
    ear_q, min_bw = 9.26449, 24.7
    idx = np.arange(n, 0, -1)
    cf = -(ear_q * min_bw) + np.exp(
        idx * (-np.log(f_high + ear_q * min_bw) + np.log(f_low + ear_q * min_bw)) / n
    ) * (f_high + ear_q * min_bw)
    return np.sort(cf)


# ---------------------------------- Simple IHC model -------------------------------------
def simple_ihc(bm_out, fs, lp_cutoff=1000, lp_order=2, compression_exp=0.3):
    """Half-wave rectification -> lowpass (loss of phase-locking) -> compressive power-law."""
    rect = np.maximum(bm_out, 0.0)
    b, a = butter(lp_order, lp_cutoff / (fs / 2))
    filtered = lfilter(b, a, rect, axis=0)
    filtered = np.maximum(filtered, 0.0)    # clip ringing before the fractional power
    return filtered ** compression_exp


# --------------------------- SACF (summary autocorrelation function) ---------------------
def autocorr_fft(x, max_lag):
    """Autocorrelation of 1-D array x (lags 0..max_lag), via the Wiener-Khinchin theorem."""
    n = len(x)
    nfft = 1
    while nfft < 2 * n:
        nfft *= 2

    # TODO, STUDENTS' TASK FOR 15 POINTS, DETAILS IN THE PDF
    #X = ...
    #acf =

    return acf[:max_lag + 1]


def compute_sacf(ihc_out, fs, f0_min=50, f0_max=800):
    """Per-channel autocorrelation, summed across channels (SACF), plus an F0 estimate
    from the strongest peak within the plausible pitch-period range."""
    nsamples, nchannels = ihc_out.shape
    max_lag = int(fs / f0_min)
    min_lag = int(fs / f0_max)

    sacf = np.zeros(max_lag + 1)
    for ch in range(nchannels):
        x = ihc_out[:, ch]  - ihc_out[:, ch].mean()
        acf = autocorr_fft(x, max_lag)
        if acf[0] > 0:
            acf = acf / acf[0]              # normalize each channel before summing
        sacf += acf

    lags_s = np.arange(max_lag + 1) / fs

    search = sacf.copy()
    search[:min_lag] = -np.inf              # exclude lag 0 and periods shorter than 1/f0_max
    peak_idx = int(np.argmax(search))
    f0_estimate = fs / peak_idx if peak_idx > 0 else np.nan

    return lags_s, sacf, f0_estimate


# ------------------------- Frame-based tracking for time-varying F0 ----------------------
def track_pitch(ihc_out, fs, frame_len_s=0.05, hop_len_s=0.01, f0_min=50, f0_max=800):
    """Slide a window over the (already-computed, continuous) IHC output and run
    compute_sacf() on each frame, to follow F0 as it changes over time -- e.g. a melody.

    The gammatone filterbank + IHC stage is run ONCE on the whole signal (it's just an
    IIR filter + static nonlinearity, so it doesn't need to be re-run per frame); only the
    SACF step -- which needs a local time window to "see" periodicity -- is framed.

    Returns
    -------
    centers_t : time (s) at the center of each analysis frame.
    f0_track : F0 estimate (Hz) for each frame.
    """
    frame_len = int(frame_len_s * fs)
    hop_len = int(hop_len_s * fs)
    n_samples = ihc_out.shape[0]

    centers_t, f0_track = [], []
    start = 0
    while start + frame_len <= n_samples:
        frame = ihc_out[start:start + frame_len, :]
        _, _, f0_est = compute_sacf(frame, fs, f0_min=f0_min, f0_max=f0_max)
        centers_t.append((start + frame_len / 2) / fs)
        f0_track.append(f0_est)
        start += hop_len

    return np.array(centers_t), np.array(f0_track)

## Demo 1: static F0, missing fundamental

A synthetic harmonic complex tone stands in for "some real musical signal" here --
swap `sig` for an actual recording (e.g. `scipy.io.wavfile.read('note.wav')`, resampled to
`fs` if needed) and everything downstream is unchanged.

Harmonics 1-2 are **deliberately left out** below to demonstrate the classic "pitch of the
missing fundamental": periodicity pitch models recover F0 from the harmonic *spacing*, not
from energy actually present at F0.

In [ ]:
fs = 16000
cf = erb_space(80, 5000, 40)              # ERB-spaced channels, 80 Hz - 5 kHz
par = initBMGTpar(cf, fs)

F0_true = 220                             # Hz
dur = 0.5                                 # s
t = np.arange(int(dur * fs)) / fs
sig = np.zeros_like(t)
for h in [3, 4, 5, 6]:                    # harmonics only -- no energy at F0 itself!
    sig += (1.0 / h) * np.sin(2 * np.pi * F0_true * h * t)
sig /= np.max(np.abs(sig))

bm_out, _ = fce_BMgammatone(sig, par)
ihc_out = simple_ihc(bm_out, fs)
lags_s, sacf, f0_est = compute_sacf(ihc_out, fs, f0_min=50, f0_max=800)
print(f"true F0 = {F0_true} Hz   |   estimated F0 = {f0_est:.1f} Hz")

max_lag = len(lags_s) - 1
acfgram = np.zeros((len(cf), max_lag + 1))
for ch in range(len(cf)):
    x = ihc_out[:, ch] - ihc_out[:, ch].mean()  # subtract the mean to suppress DC component
    a = autocorr_fft(x, max_lag)
    acfgram[ch] = a / a[0] if a[0] > 0 else a

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].imshow(acfgram, aspect='auto', origin='lower',
               extent=[lags_s[0]*1000, lags_s[-1]*1000, 0, len(cf)], cmap='viridis')
axes[0].set_ylabel('Channel (low -> high CF)')
axes[0].set_title('Autocorrelogram (per-channel ACF)')

axes[1].plot(lags_s * 1000, sacf)
axes[1].axvline(1000 / f0_est, color='crimson', ls='--', label=f'detected F0 = {f0_est:.1f} Hz')
axes[1].axvline(1000 / F0_true, color='gray', ls=':', label=f'true F0 = {F0_true} Hz')
axes[1].set_xlabel('Lag (ms)')
axes[1].set_ylabel('SACF')
axes[1].legend()
plt.tight_layout()
plt.show()

## Demo 2: a melody (time-varying F0)

Same filterbank + IHC output as always, but now `track_pitch()` slides a short analysis
window across it and re-runs the SACF step on each window, producing a pitch *contour*
instead of a single number -- this is the part you'd need for a real recording of someone
singing or playing an instrument.

This time the full harmonic series (fundamental included) is used, rather than the
missing-fundamental stimulus from Demo 1 -- right at a note transition, the analysis window
briefly contains two different pitches at once, and asking the model to resolve that from
weak high-harmonic-only periodicity is a much harder (and noisier) problem than the
tracking point this demo is making. Feel free to try `harmonics=(3,4,5,6)` here afterwards
and see for yourselves how much less stable the transitions become.

Two parameters control the time/frequency trade-off: `frame_len_s` must be long enough to
contain a few periods of the lowest F0 you want to detect (period = 1/`f0_min`), while
`hop_len_s` sets how often you get a new estimate. Note the smearing visible right at each
note transition below -- that's the window straddling two different periods at once, and is
worth discussing in class as a fundamental limit of short-time periodicity analysis.

In [ ]:
notes_hz = [262, 294, 330, 349, 392, 349, 330, 294]   # a short C-D-E-F-G-F-E-D melody
note_dur = 0.25                                       # s per note


def note_freqs_in_range(fmin, fmax, A4=440.0):
    """All semitone frequencies (+labels, +whether natural/white-key) spanning [fmin, fmax],
    used as a musical-note reference grid on a pitch plot."""
    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    midi_min = int(np.floor(69 + 12 * np.log2(fmin / A4))) - 1
    midi_max = int(np.ceil(69 + 12 * np.log2(fmax / A4))) + 1
    freqs, labels, is_natural = [], [], []
    for m in range(midi_min, midi_max + 1):
        name = note_names[m % 12]
        freqs.append(A4 * 2 ** ((m - 69) / 12))
        labels.append(f"{name}{m // 12 - 1}")
        is_natural.append('#' not in name)
    return np.array(freqs), np.array(labels), np.array(is_natural)


def note_signal(f0, dur, fs, harmonics=(1, 2, 3, 4, 5, 6)):
    """One note: a harmonic complex tone with a short raised-cosine envelope so notes
    don't click at the boundaries."""
    t = np.arange(int(dur * fs)) / fs
    sig = np.zeros_like(t)
    for h in harmonics:
        sig += (1.0 / h) * np.sin(2 * np.pi * f0 * h * t)
    ramp_n = int(0.01 * fs)
    env = np.ones_like(t)
    ramp = 0.5 * (1 - np.cos(np.pi * np.arange(ramp_n) / ramp_n))
    env[:ramp_n] = ramp
    env[-ramp_n:] = ramp[::-1]
    return sig * env

melody = np.concatenate([note_signal(f0, note_dur, fs) for f0 in notes_hz])
melody /= np.max(np.abs(melody))
t_melody = np.arange(len(melody)) / fs

# ground-truth step contour, just for plotting comparison
true_t, true_f0 = [], []
for i, f0 in enumerate(notes_hz):
    true_t += [i * note_dur, (i + 1) * note_dur]
    true_f0 += [f0, f0]

bm_out, _ = fce_BMgammatone(melody, par)             # filterbank run ONCE, on the whole melody
ihc_out = simple_ihc(bm_out, fs)
centers_t, f0_track = track_pitch(ihc_out, fs, frame_len_s=0.05, hop_len_s=0.01)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(t_melody, melody, lw=0.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Melody waveform (synthetic, harmonic notes)')

fmin = min(np.min(true_f0), np.nanmin(f0_track))
fmax = max(np.max(true_f0), np.nanmax(f0_track))
note_f, note_lbl, is_nat = note_freqs_in_range(fmin, fmax)

ax_notes = axes[1]
ax_notes.semilogy(true_t, true_f0, color='gray', ls=':', lw=2, label='true F0 (note sequence)')
ax_notes.semilogy(centers_t, f0_track, color='crimson', marker='.', ms=4, lw=1, label='SACF-estimated F0')
for f, nat in zip(note_f, is_nat):
    ax_notes.axhline(f, color='gray', lw=0.8 if nat else 0.3, alpha=0.5 if nat else 0.2, zorder=0)
ax_notes.yaxis.set_major_locator(mticker.FixedLocator(note_f[is_nat]))
ax_notes.yaxis.set_major_formatter(mticker.FixedFormatter(note_lbl[is_nat]))
ax_notes.yaxis.set_minor_locator(mticker.NullLocator())
ax_notes.set_ylim(fmin * 0.95, fmax * 1.05)
ax_notes.set_xlabel('Time (s)')
ax_notes.set_ylabel('Estimated pitch (note)')
ax_notes.legend()

ax_hz = ax_notes.secondary_yaxis('right')
ax_hz.set_ylabel('Frequency (Hz)')
ax_hz.yaxis.set_major_locator(mticker.LogLocator(base=10, subs=np.arange(1, 10)))
ax_hz.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax_hz.yaxis.set_minor_formatter(mticker.NullFormatter())

plt.tight_layout()
plt.show()

## Demo 3: streaming / causal tracking (constant memory)

`track_pitch()` above needs the *entire* `ihc_out` array in memory before it can start --
it was computed by filtering the whole signal in one call. Memory for that scales with
signal duration: O(n_samples x n_channels). For a long recording (or a live microphone
feed, where "the whole signal" doesn't exist yet) that's wasteful or outright impossible.

`fce_BMgammatone()` already supports carrying its filter state between calls (the
`pastGTn` argument) -- it just wasn't being used that way before. Giving the IHC lowpass
stage the same treatment (persisting its own filter state across chunks, instead of
re-filtering from scratch) lets the whole pipeline run **hop-by-hop**, keeping only a
rolling buffer of the last `frame_len` IHC samples -- memory now stays constant,
O(frame_len x n_channels), no matter how long the signal is.

In [ ]:
def track_pitch_streaming(signal, par, fs, frame_len_s=0.05, hop_len_s=0.01,
                           lp_cutoff=1000, lp_order=2, compression_exp=0.3,
                           f0_min=50, f0_max=800):
    """Causal pitch tracker: processes the signal hop-by-hop, keeping only a rolling
    buffer of IHC output (not the whole signal) in memory. Gives IDENTICAL results to
    track_pitch(full_ihc_out, ...), just with constant memory use instead of O(duration)."""
    frame_len = int(frame_len_s * fs)
    hop_len = int(hop_len_s * fs)
    n_channels = len(par.cfArray)
    lp_b, lp_a = butter(lp_order, lp_cutoff / (fs / 2))

    gt_state = None                                  # gammatone filterbank state
    ihc_state = np.zeros((lp_order, n_channels))      # IHC lowpass state
    ihc_buffer = np.zeros((frame_len, n_channels))    # the ONLY per-sample buffer kept

    centers_t, f0_track = [], []
    n_samples = len(signal)
    start = 0
    while start + hop_len <= n_samples:
        chunk = signal[start:start + hop_len]

        bm_chunk, gt_state = fce_BMgammatone(chunk, par, gt_state)
        rect_chunk = np.maximum(bm_chunk, 0.0)
        lp_chunk, ihc_state = lfilter(lp_b, lp_a, rect_chunk, axis=0, zi=ihc_state)
        lp_chunk = np.maximum(lp_chunk, 0.0)
        ihc_chunk = lp_chunk ** compression_exp

        ihc_buffer = np.vstack([ihc_buffer[hop_len:], ihc_chunk])   # slide the window

        if start + hop_len >= frame_len:              # buffer is fully "real" data by now
            _, _, f0_est = compute_sacf(ihc_buffer, fs, f0_min=f0_min, f0_max=f0_max)
            centers_t.append((start + hop_len - frame_len / 2) / fs)
            f0_track.append(f0_est)
        start += hop_len

    return np.array(centers_t), np.array(f0_track)


centers_stream, f0_stream = track_pitch_streaming(melody, par, fs)

fmin = min(np.min(true_f0), np.nanmin(f0_track), np.nanmin(f0_stream))
fmax = max(np.max(true_f0), np.nanmax(f0_track), np.nanmax(f0_stream))
note_f, note_lbl, is_nat = note_freqs_in_range(fmin, fmax)

fig, ax_notes = plt.subplots(figsize=(9, 4.5))
ax_notes.semilogy(true_t, true_f0, color='gray', ls=':', lw=2, label='true F0')
ax_notes.semilogy(centers_t, f0_track, color='crimson', lw=1, label='batch track_pitch()')
ax_notes.semilogy(centers_stream, f0_stream, color='navy', ls='--', lw=1, label='streaming track_pitch_streaming()')
for f, nat in zip(note_f, is_nat):
    ax_notes.axhline(f, color='gray', lw=0.8 if nat else 0.3, alpha=0.5 if nat else 0.2, zorder=0)
ax_notes.yaxis.set_major_locator(mticker.FixedLocator(note_f[is_nat]))
ax_notes.yaxis.set_major_formatter(mticker.FixedFormatter(note_lbl[is_nat]))
ax_notes.yaxis.set_minor_locator(mticker.NullLocator())
ax_notes.set_ylim(fmin * 0.95, fmax * 1.05)
ax_notes.set_xlabel('Time (s)')
ax_notes.set_ylabel('Pitch (note)')
ax_notes.legend()
ax_notes.set_title('Batch vs. streaming -- should overlap exactly')

ax_hz = ax_notes.secondary_yaxis('right')
ax_hz.set_ylabel('Frequency (Hz)')
ax_hz.yaxis.set_major_locator(mticker.LogLocator(base=10, subs=np.arange(1, 10)))
ax_hz.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax_hz.yaxis.set_minor_formatter(mticker.NullFormatter())

plt.tight_layout()
plt.show()

# memory, not just accuracy, is the actual point here
for duration_s in [2, 30, 180]:
    n_samples = duration_s * fs
    n_channels = len(cf)
    batch_mb = n_samples * n_channels * 8 * 2 / 1e6       # bm_out + ihc_out, float64
    stream_mb = int(0.05 * fs) * n_channels * 8 * 2 / 1e6 # ihc_buffer, roughly
    print(f"{duration_s:4d} s signal:  batch ~ {batch_mb:8.1f} MB   |   streaming ~ {stream_mb:6.3f} MB")

### Process recorded wav file with voice signing

Two wav files with female voice signing are provided in github

In [ ]:
import urllib.request
from scipy.io import wavfile
from scipy.signal import resample_poly
from math import gcd
import matplotlib.ticker as mticker


def load_wav_for_model(path, target_fs):
    """Load a WAV file, convert to mono float [-1, 1], and resample to the
    filterbank's working sample rate `target_fs` if the file's native rate differs."""
    native_fs, raw = wavfile.read(path)
    is_int = np.issubdtype(raw.dtype, np.integer)
    data = raw.astype(np.float64)
    if data.ndim > 1:
        data = data.mean(axis=1)                         # stereo -> mono
    if is_int:
        data = data / np.iinfo(raw.dtype).max             # e.g. int16 PCM -> [-1, 1]
    else:
        peak = np.max(np.abs(data))
        if peak > 0:
            data = data / peak
    if native_fs != target_fs:
        g = gcd(int(native_fs), int(target_fs))
        data = resample_poly(data, target_fs // g, native_fs // g)   # anti-aliased resampling
    return data


def frame_rms(signal, fs, frame_len_s=0.05, hop_len_s=0.01):
    """RMS energy per analysis frame, aligned with track_pitch_streaming()'s frame centers --
    used below as a simple voicing/silence gate."""
    frame_len = int(frame_len_s * fs)
    hop_len = int(hop_len_s * fs)
    n_samples = len(signal)
    rms = []
    start = 0
    while start + hop_len <= n_samples:
        if start + hop_len >= frame_len:
            seg = signal[start + hop_len - frame_len:start + hop_len]
            rms.append(np.sqrt(np.mean(seg ** 2)))
        start += hop_len
    return np.array(rms)


# --- download a real recording from GitHub and track its pitch ---------------------------
# you can comment one of these
GITHUB_RAW = ('https://raw.githubusercontent.com/vencov/FAV_upsidedown/main/TutSACF03/'
              '417941__thescarlettwitch89__i-hang-up-when-you-call-vocal-sample-3.wav')
#'I hang up when you call' vocal sample (3).m4a by TheScarlettWitch89 -- https://freesound.org/s/417941/ -- License: Attribution 3.0

#GITHUB_RAW = ('https://raw.githubusercontent.com/vencov/FAV_upsidedown/main/TutSACF03/'
#              '625399__drotzruhn__female-singing-ooo.wav')
# Female singing Ooo by drotzruhn -- https://freesound.org/s/625399/ -- License: Attribution 4.0

local_path = 'vocal_sample.wav'
urllib.request.urlretrieve(GITHUB_RAW, local_path)

sig = load_wav_for_model(local_path, fs)
t_sig = np.arange(len(sig)) / fs

centers_t, f0_track = track_pitch_streaming(sig, par, fs, f0_min=70, f0_max=500)

# real speech has silence and unvoiced sounds with no real periodicity -- without gating,
# SACF just reports whatever noise peak happens to be largest there. A simple relative
# energy threshold masks those frames out instead of showing meaningless "pitch".
rms = frame_rms(sig, fs)

voiced = rms > 0.05 * rms.max()  # manually can chnage the threshold
f0_track = np.where(voiced, f0_track, np.nan)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t_sig, sig, lw=0.3, color='steelblue')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Loaded recording (from GitHub)')

valid = ~np.isnan(f0_track)
fmin, fmax = np.nanmin(f0_track[valid]), np.nanmax(f0_track[valid])
note_f, note_lbl, is_nat = note_freqs_in_range(fmin, fmax)

ax_notes = axes[1]
ax_notes.semilogy(centers_t, f0_track, color='crimson', marker='.', ms=3, lw=0.8)
for f, nat in zip(note_f, is_nat):
    ax_notes.axhline(f, color='gray', lw=0.8 if nat else 0.3, alpha=0.6 if nat else 0.25, zorder=0)
ax_notes.yaxis.set_major_locator(mticker.FixedLocator(note_f[is_nat]))
ax_notes.yaxis.set_major_formatter(mticker.FixedFormatter(note_lbl[is_nat]))
ax_notes.yaxis.set_minor_locator(mticker.NullLocator())
ax_notes.set_ylim(fmin * 0.9, fmax * 1.1)
ax_notes.set_xlabel('Time (s)')
ax_notes.set_ylabel('Estimated pitch (note)')

# second y-axis, same data/scale, plain Hz instead of note names
ax_hz = ax_notes.secondary_yaxis('right')
ax_hz.set_ylabel('Frequency (Hz)')
ax_hz.yaxis.set_major_locator(mticker.LogLocator(base=10, subs=np.arange(1, 10)))
ax_hz.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax_hz.yaxis.set_minor_formatter(mticker.NullFormatter())

plt.tight_layout()
plt.show()